# REDLAMP Gradient Conflict Profiling (Spec Locked)
- Log mỗi iteration.
- Đo toàn bộ encoder.
- Layer trọng tâm: bottleneck/projection layer.
- Optimizer step theo `L_total.backward()`.
- Lưu `raw + EMA(alpha=0.1) + SMA(window=50)`.


In [ ]:
import copy
import collections
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=4, stride=2, padding=2)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(self.bn(self.conv(x))))


In [ ]:
class RedLampEncoder(nn.Module):
    def __init__(self, in_ch=1, filters=(64, 64, 128, 128), embed_dim=128, dropout=0.2):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(in_ch,      filters[0], dropout),
            ConvBlock(filters[0], filters[1], dropout),
            ConvBlock(filters[1], filters[2], dropout),
            ConvBlock(filters[2], filters[3], dropout),
        )
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.proj = nn.Conv1d(filters[3], embed_dim, kernel_size=1)

    def forward(self, x):
        x = self.blocks(x)
        x = self.pool(x)
        x = self.proj(x)
        return x


In [ ]:
x = torch.randn(128, 1, 20)
enc = RedLampEncoder()
z = enc(x)
print('input:', x.shape)
print('latent:', z.shape)


In [ ]:
class TinyMTL(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()
        self.encoder = RedLampEncoder()
        self.cls = nn.Sequential(nn.Flatten(), nn.LazyLinear(64), nn.ReLU(), nn.Linear(64, num_classes))
        self.dec = nn.Sequential(nn.ConvTranspose1d(128, 64, kernel_size=3, padding=1), nn.ReLU(), nn.Conv1d(64, 1, kernel_size=1))

    def forward(self, x):
        z = self.encoder(x)
        logits = self.cls(z)
        recon = self.dec(z)
        recon = F.interpolate(recon, size=x.shape[-1], mode='linear', align_corners=False)
        return z, logits, recon


In [ ]:
def masked_mse_anomaly_free(pred, target, anomaly_mask):
    # anomaly_mask: [B, L], 1=anomaly, 0=normal
    normal = (1.0 - anomaly_mask.float()).unsqueeze(1)
    sq = (pred - target) ** 2
    denom = normal.sum() * pred.shape[1]
    if denom.item() <= 0:
        return sq.mean()
    return (sq * normal).sum() / denom


In [ ]:
def ema_update(prev, cur, alpha=0.1):
    if prev is None:
        return cur
    return alpha * cur + (1.0 - alpha) * prev


def sma_update(queue, cur, window=50):
    queue.append(cur)
    return sum(queue) / len(queue)


In [ ]:
def list_encoder_layers_for_grad(model):
    layers = []
    for name, p in model.encoder.named_parameters():
        if p.requires_grad:
            layers.append(name)
    return layers


def collect_weighted_grads(loss, params, retain_graph=False):
    grads = torch.autograd.grad(loss, params, retain_graph=retain_graph, allow_unused=True)
    out = {}
    for n, g in grads:
        out[n] = None if g is None else g.detach().clone()
    return out


def flatten_or_zero(g, ref):
    if g is None:
        return torch.zeros_like(ref).reshape(-1)
    return g.reshape(-1)


In [ ]:
def profile_step(model, optimizer, x, y_cls, anomaly_mask, gamma=0.1, ema=None, sma=None, ema_alpha=0.1, sma_window=50):
    if ema is None:
        ema = {}
    if sma is None:
        sma = {}

    z, logits, recon = model(x)
    l_ce = F.cross_entropy(logits, y_cls)
    l_mse = masked_mse_anomaly_free(recon, x, anomaly_mask)

    l_ce_w = gamma * l_ce
    l_mse_w = (1.0 - gamma) * l_mse
    l_total = l_ce_w + l_mse_w

    named_params = [(n, p) for n, p in model.encoder.named_parameters() if p.requires_grad]

    g_ce = collect_weighted_grads(l_ce_w, named_params, retain_graph=True)
    g_mse = collect_weighted_grads(l_mse_w, named_params, retain_graph=True)
    g_total_analytic = {n: (torch.zeros_like(p) if g_ce[n] is None else g_ce[n]) + (torch.zeros_like(p) if g_mse[n] is None else g_mse[n]) for n, p in named_params}

    optimizer.zero_grad(set_to_none=True)
    l_total.backward()

    logs = {}
    for n, p in named_params:
        g_ce_f = flatten_or_zero(g_ce[n], p)
        g_mse_f = flatten_or_zero(g_mse[n], p)
        g_tot_f = g_total_analytic[n].reshape(-1)

        ce_norm = g_ce_f.norm().item()
        mse_norm = g_mse_f.norm().item()
        tot_norm = g_tot_f.norm().item()

        denom = (ce_norm * mse_norm) + 1e-12
        cosine = float(torch.dot(g_ce_f, g_mse_f).item() / denom)
        r_ratio = float(tot_norm / (ce_norm + mse_norm + 1e-12))

        # Validation: tổng analytic phải khớp grad autograd của L_total
        assert torch.allclose(g_total_analytic[n], p.grad.detach(), atol=1e-6, rtol=1e-4), f'grad mismatch at {n}'

        key_c = f'cos/{n}'
        key_r = f'rr/{n}'
        logs[key_c] = cosine
        logs[key_r] = r_ratio

        ema[key_c] = ema_update(ema.get(key_c), cosine, alpha=ema_alpha)
        ema[key_r] = ema_update(ema.get(key_r), r_ratio, alpha=ema_alpha)

        if key_c not in sma:
            sma[key_c] = collections.deque(maxlen=sma_window)
        if key_r not in sma:
            sma[key_r] = collections.deque(maxlen=sma_window)
        logs[f'ema/{key_c}'] = ema[key_c]
        logs[f'ema/{key_r}'] = ema[key_r]
        logs[f'sma/{key_c}'] = sma_update(sma[key_c], cosine, window=sma_window)
        logs[f'sma/{key_r}'] = sma_update(sma[key_r], r_ratio, window=sma_window)

    optimizer.step()
    logs['loss/ce'] = float(l_ce.item())
    logs['loss/mse'] = float(l_mse.item())
    logs['loss/total'] = float(l_total.item())
    return logs, ema, sma


In [ ]:
model = TinyMTL(num_classes=12)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('Encoder layers that can be profiled:')
for n in list_encoder_layers_for_grad(model):
    print('-', n)

focus_layer = 'proj.weight'
print('
Focus layer (bottleneck/projection):', focus_layer)


In [ ]:
ema_state, sma_state = {}, {}
all_logs = []

for step in range(5):
    x = torch.randn(16, 1, 20)
    y = torch.randint(0, 12, (16,))
    m = (torch.rand(16, 20) < 0.2).long()

    logs, ema_state, sma_state = profile_step(
        model, optimizer, x, y, m,
        gamma=0.1,
        ema=ema_state,
        sma=sma_state,
        ema_alpha=0.1,
        sma_window=50,
    )
    all_logs.append(logs)

    print(f"step={step+1} total={logs['loss/total']:.4f} "
          f"raw_cos={logs['cos/proj.weight']:.4f} "
          f"ema_cos={logs['ema/cos/proj.weight']:.4f} "
          f"sma_cos={logs['sma/cos/proj.weight']:.4f}")
